In [13]:
import os
import json

from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveJsonSplitter
from langchain_core.documents import Document
import gradio as gr

# import gradio as gr

load_dotenv(override=True)

True

In [14]:

print(os.getenv("API_KEY"))


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


In [15]:
print("Environment variables loaded.")

Environment variables loaded.


In [16]:

with open('valorant_ldoc.json') as f:
    data = json.load(f)

splitter = RecursiveJsonSplitter(max_chunk_size=1000)
json_chunks = splitter.split_json(data)  # Yields list of dict chunks


In [17]:
documents = [
    Document(
        page_content=json.dumps(chunk, ensure_ascii=False),
        metadata={"source": "valorant_ldoc"}
    )
    for chunk in json_chunks
]

####rag check

In [18]:
# ## relaoding the vectordb without re-embedding
# from langchain_huggingface import HuggingFaceEmbeddings
# embedding = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-large-en-v1.5"
# )

# vectordb = Chroma(
#     persist_directory="./vector_db",
#     embedding_function=embedding
# )

In [20]:
# embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
embeddings = HuggingFaceEmbeddings(model_name = "BAAI/bge-large-en-v1.5")

db_name = "val_vector_db"

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
    
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings, persist_directory=db_name)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 708.49it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

In [ ]:
retreiver = vectordb.as_retriever(search_type="similarity",search_kwargs={"k": 5})

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="tngtech/tng-r1t-chimera:free",  # example
    openai_api_key=os.getenv("API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.1,
    top_p=0.9,
    max_tokens=512,
)

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are an AI Scouting Analyst for professional VALORANT esports.

Your task is to generate concise, data-grounded scouting insights about an upcoming opponent
using ONLY the provided retrieved context from official match data.

STRICT RULES:
1. You must rely exclusively on the retrieved context.
2. Do NOT use outside knowledge, assumptions, or general VALORANT meta knowledge.
3. If the context does not contain enough information to answer a question, explicitly say:
   "Insufficient data available in the provided matches."
4. Do NOT speculate, predict outcomes, or invent tendencies.
5. Do NOT generalize beyond the scope of the provided series or maps.

OUTPUT STYLE:
- Write in clear, professional analyst language.
- Prefer bullet points over paragraphs.
- Be factual, neutral, and concise.
- Avoid hype, opinions, or narrative storytelling.

ALLOWED INSIGHTS (only if supported by context):
- Map-specific tendencies
- Agent compositions and pick patterns
- Player agent usage and consistency
- Observed attack or defense preferences
- Repeated behaviors across rounds or maps

DISALLOWED CONTENT:
- Predictions or win probabilities
- Coaching advice not supported by data
- Claims like "always", "never", or "dominant"
- Long-term trends beyond the given data
- Subjective judgments (e.g., "strong", "weak")

STRUCTURE YOUR RESPONSE AS:
- Section headers (e.g., "Map Tendencies", "Player Tendencies")
- Bullet points under each section
- A short "Key Takeaways" section with 2–3 bullets max

Remember:
Accuracy and restraint are more important than completeness.

"""

In [ ]:
def answer_question(question: str, history):
    docs = retreiver.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Did any player switch roles or agents across maps?", [])

'\n\n**Player Role/Agent Consistency**  \n- Insufficient data available in the provided matches.  \n\n**Key Takeaways**  \n- No agent or role-switching patterns can be identified from the provided match data.  \n- Context lacks specific player agent selections across maps.'

In [ ]:
gr.ChatInterface(answer_question).launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2f6ed7c581fc95cf71.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
